# Grito Labs — Marketing Analytics & Attribution

Synthetic dataset demonstration.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("../data")
daily = pd.read_csv(DATA/"marketing_daily.csv", parse_dates=["date"])
touchpoints = pd.read_csv(DATA/"touchpoints.csv", parse_dates=["timestamp"])
conversions = pd.read_csv(DATA/"conversions.csv", parse_dates=["conversion_timestamp"])


## KPI calculation

In [ ]:
spend = daily["spend"].sum()
impressions = daily["impressions"].sum()
clicks = daily["clicks"].sum()
revenue = conversions["revenue"].sum()
conversion_count = conversions["conversion_id"].nunique()

kpis = {
    "Spend": spend,
    "Revenue": revenue,
    "Conversions": conversion_count,
    "CTR": clicks / impressions,
    "CPC": spend / clicks,
    "AOV": revenue / conversion_count,
}
pd.Series(kpis)


## Channel performance

In [ ]:
channel = daily.groupby("channel").agg(
    spend=("spend","sum"),
    impressions=("impressions","sum"),
    clicks=("clicks","sum")
)
channel["ctr"] = channel["clicks"] / channel["impressions"]
channel["cpc"] = channel["spend"] / channel["clicks"]
channel.sort_values("spend", ascending=False)


## Attribution

The exported `attribution_touchpoints.csv` contains weights for first-touch, last-touch, linear and position-based attribution.

In [ ]:
attr = pd.read_csv(DATA/"attribution_touchpoints.csv")
for model, weight in {
    "First Touch":"first_touch_weight",
    "Last Touch":"last_touch_weight",
    "Linear":"linear_weight",
    "Position Based":"position_weight"
}.items():
    x = attr.assign(attributed_revenue=attr["revenue"]*attr[weight]).groupby("channel")["attributed_revenue"].sum()
    print("\n", model)
    display(x.sort_values(ascending=False))


## Business interpretation

Use the attribution comparison to avoid making budget decisions from a single credit-assignment rule. Validate major reallocations with incrementality testing.